# Serial Frames for IRX4 Plus Multi Protocol Transmitter

## Flysky Protocol (AFHDS 1A)

### Simulate Channel 1

In [41]:
import time
import serial
import sys

def pack_channels_11bit(ch):
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,
    rx_num=0,
    type_value=0,
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind: b1 |= 0x80
    if autobind: b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power: b2 |= 0x80

    b3 = option_value & 0xFF

    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    hexstr = " ".join(f"{b:02X}" for b in frame)
    print(f"{label} ({len(frame)} bytes):")
    print(hexstr)


# ------------------------
# Serial setup
# ------------------------

PORT = "/dev/ttyUSB0"
BAUD = 100000

PROTO_FLYSKY = 1
FLYSKY_SUBTYPE = 0

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

print("Streaming… press Ctrl+C to stop")

channels = [1024] * 16
frame_hz = 50
period = 1.0 / frame_hz
t0 = time.time()


# Print exactly once
frame = make_multi_v1_frame_channels(
    sub_protocol=PROTO_FLYSKY,
    type_value=FLYSKY_SUBTYPE,
    channels=channels
)
print_frame_hex_once(frame)

try:
    while True:
        # simple CH1 sweep
        elapsed = time.time() - t0
        phase = (elapsed % 2.0) / 2.0
        lower = 500
        upper = 1500
        #channels[0] = int(204 + phase * (1843 - 204))
        channels[0] = int(lower + phase * (upper - lower))

        frame = make_multi_v1_frame_channels(
            sub_protocol=PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE,
            channels=channels
        )

        ser.write(frame)
        time.sleep(period)

except KeyboardInterrupt:
    print("\nCtrl+C received — stopping transmitter")

finally:
    # optional: send neutral frames before exit
    neutral = [1024] * 16
    for _ in range(3):
        ser.write(make_multi_v1_frame_channels(
            PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE,
            channels=neutral
        ))
        time.sleep(0.02)

    ser.close()
    print("Serial port closed cleanly")


Streaming… press Ctrl+C to stop
TX frame (26 bytes):
55 01 00 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80

Ctrl+C received — stopping transmitter
Serial port closed cleanly


In [9]:
import serial, time
ser = serial.Serial("/dev/ttyUSB0", 100000, bytesize=8, parity='E', stopbits=2)
try:
    while True:
        ser.write(b"\x55\xAA\x00\xFF")   # recognizable pattern
        time.sleep(0.01)
except KeyboardInterrupt:
    pass
finally:
    ser.close()


### Bind

In [32]:
# ------------------------
# Serial setup
# ------------------------

PORT = "/dev/ttyUSB0"
BAUD = 100000

PROTO_FLYSKY = 1
FLYSKY_SUBTYPE = 0

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

print("Binding… press Ctrl+C to abort")

try:
    # ---- Bind phase ----
    start = time.time()
    while time.time() - start < 6.0:
        ser.write(make_multi_v1_frame_channels(
            sub_protocol=PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE,
            bind=True
        ))
        time.sleep(0.02)

    print("Bind done → normal mode")

    # ---- Normal mode ----
    while True:
        ser.write(make_multi_v1_frame_channels(
            sub_protocol=PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE
        ))
        time.sleep(0.02)

except KeyboardInterrupt:
    print("\nCtrl+C — exiting")

finally:
    ser.close()
    print("Serial closed")


Binding… press Ctrl+C to abort
Bind done → normal mode

Ctrl+C — exiting
Serial closed


## AFHDS2A Protocol

### Bind

In [31]:
import time
import serial

def pack_channels_11bit(ch):
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,
    rx_num=0,
    type_value=0,
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF
    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    print(f"{label} ({len(frame)} bytes):")
    print(" ".join(f"{b:02X}" for b in frame))

# Serial (MULTI serial mode: 100000 8E2)
PORT = "/dev/ttyUSB0"
BAUD = 100000
ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

PROTO_AFHDS2A = 28
AFHDS2A_PPM_IBUS = 1  # :contentReference[oaicite:8]{index=8}

# AFHDS2A binding note: use a different rx_num for each receiver you bind :contentReference[oaicite:9]{index=9}
rx_num = 0 #1 #0            # try 0..15 (and try another number if bind fails or if reusing an old bind)
option_value = 10      # 0=50Hz :contentReference[oaicite:10]{index=10}

frame_hz = 50
period = 1.0 / frame_hz
channels = [1024] * 16

print("AFHDS2A bind → run… Ctrl+C to stop")

try:
    # --- Binding phase ---
    bind_seconds = 6.0
    t0 = time.time()

    bind_frame = make_multi_v1_frame_channels(
        sub_protocol=PROTO_AFHDS2A,
        rx_num=rx_num,
        type_value=AFHDS2A_PPM_IBUS,
        option_value=option_value,
        bind=True,
        channels=channels
    )
    print_frame_hex_once(bind_frame, label="BIND frame (first one)")

    while time.time() - t0 < bind_seconds:
        ser.write(bind_frame)
        time.sleep(period)

    print("Bind phase complete. Switching to normal mode.")

    # --- Normal run phase (CH1 sweep) ---
    t1 = time.time()
    while True:
        elapsed = time.time() - t1
        phase = (elapsed % 2.0) / 2.0
        channels[0] = int(204 + phase * (1843 - 204))

        frame = make_multi_v1_frame_channels(
            sub_protocol=PROTO_AFHDS2A,
            rx_num=rx_num,
            type_value=AFHDS2A_PPM_IBUS,
            option_value=option_value,
            bind=False,
            channels=channels
        )
        ser.write(frame)
        time.sleep(period)

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    ser.close()
    print("Serial closed.")


AFHDS2A bind → run… Ctrl+C to stop
BIND frame (first one) (26 bytes):
55 9C 10 0A 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80
Bind phase complete. Switching to normal mode.

Stopped (Ctrl+C).
Serial closed.


In [26]:
import time
import serial

def pack_channels_11bit(ch):
    """Pack 16x 11-bit channel values into 22 bytes (SBUS-style)."""
    if len(ch) != 16:
        raise ValueError("Need exactly 16 channels")
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        if not (0 <= v <= 2047):
            raise ValueError("Channel values must be 0..2047")
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,          # 0..31 => header 0x55
    rx_num=0,              # important for AFHDS2A: different RXs must use different rx_num :contentReference[oaicite:3]{index=3}
    type_value=0,          # sub-protocol (0..7) placed in Stream[2] bits 4..6
    option_value=0,        # AFHDS2A: refresh rate option (0=50Hz, 70=400Hz) :contentReference[oaicite:4]{index=4}
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF  # signed int8 on wire, but ok to send as byte

    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    print(f"{label} ({len(frame)} bytes):")
    print(" ".join(f"{b:02X}" for b in frame))

# ------------------------
# Serial setup (MULTI = 100000 baud, 8E2)
# ------------------------
PORT = "/dev/ttyUSB0"   # change if needed
BAUD = 100000

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

# ------------------------
# AFHDS2A settings
# ------------------------
PROTO_AFHDS2A = 28

# Sub-protocols (from MULTI docs):
# 0 PWM_IBUS
# 1 PPM_IBUS  <-- pick this for "PPM" variant
# 2 PWM_SBUS
# 3 PPM_SBUS
# 4 PWM_IBUS16
# 5 PPM_IBUS16
AFHDS2A_PPM_IBUS = 1  # :contentReference[oaicite:5]{index=5}

rx_num = 0            # try 0..15; also try different numbers if rebinding multiple RXs :contentReference[oaicite:6]{index=6}
option_value = 0      # 0=50Hz (safe start) :contentReference[oaicite:7]{index=7}

channels = [1024] * 16
frame_hz = 50
period = 1.0 / frame_hz
t0 = time.time()

print("AFHDS2A normal streaming… Ctrl+C to stop")

try:
    # Print one sample frame (once)
    channels[0] = 1024
    sample = make_multi_v1_frame_channels(
        sub_protocol=PROTO_AFHDS2A,
        rx_num=rx_num,
        type_value=AFHDS2A_PPM_IBUS,
        option_value=option_value,
        bind=False,
        channels=channels
    )
    print_frame_hex_once(sample, label="Sample AFHDS2A frame")

    # Stream continuously; modulate CH1 only
    while True:
        elapsed = time.time() - t0
        phase = (elapsed % 2.0) / 2.0     # 0..1 over 2 seconds
        channels[0] = int(204 + phase * (1843 - 204))  # ~-100%..+100%

        frame = make_multi_v1_frame_channels(
            sub_protocol=PROTO_AFHDS2A,
            rx_num=rx_num,
            type_value=AFHDS2A_PPM_IBUS,
            option_value=option_value,
            bind=False,
            channels=channels
        )
        ser.write(frame)
        time.sleep(period)

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    # send a couple neutral frames then close
    neutral = [1024] * 16
    for _ in range(3):
        ser.write(make_multi_v1_frame_channels(
            PROTO_AFHDS2A, rx_num=rx_num, type_value=AFHDS2A_PPM_IBUS,
            option_value=option_value, bind=False, channels=neutral
        ))
        time.sleep(0.02)
    ser.close()
    print("Serial closed.")


AFHDS2A normal streaming… Ctrl+C to stop
Sample AFHDS2A frame (26 bytes):
55 1C 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80

Stopped (Ctrl+C).
Serial closed.


In [33]:
import time
import serial

def pack_channels_11bit(ch):
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        if not (0 <= v <= 2047):
            raise ValueError("Channel values must be 0..2047")
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,
    rx_num=0,
    type_value=0,
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF
    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    print(f"{label} ({len(frame)} bytes):")
    print(" ".join(f"{b:02X}" for b in frame))

# Serial: 100000 8E2
PORT = "/dev/ttyUSB0"
BAUD = 100000
ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

PROTO_AFHDS2A = 28

# Try the common AFHDS2A variants (many receivers don’t care, but some do)
SUBPROTOS = [
    ("PPM_IBUS", 1),
    ("PWM_IBUS", 0),
    ("PPM_SBUS", 3),
    ("PWM_SBUS", 2),
    ("PPM_IBUS16", 5),
    ("PWM_IBUS16", 4),
]

# Try several rx numbers (AFHDS2A uses receiver match; some setups expect non-zero)
RX_NUMS = list(range(0, 8))  # 0..7

option_value = 0
frame_hz = 50
period = 1.0 / frame_hz
channels = [1024] * 16

print("AFHDS2A binding scan… put RX in bind mode, then run this. Ctrl+C to stop.")

try:
    # For each combo: send bind frames for a few seconds
    bind_seconds_each = 5.0

    first = True
    for (name, type_value) in SUBPROTOS:
        for rx_num in RX_NUMS:
            print(f"\nTrying AFHDS2A bind: sub={name}({type_value}) rx_num={rx_num} (sending {bind_seconds_each}s)")

            t0 = time.time()
            while time.time() - t0 < bind_seconds_each:
                frame = make_multi_v1_frame_channels(
                    sub_protocol=PROTO_AFHDS2A,
                    rx_num=rx_num,
                    type_value=type_value,
                    option_value=option_value,
                    bind=True,
                    autobind=True,   # many people rely on this for AFHDS2A-style bind flows
                    channels=channels
                )
                if first:
                    print_frame_hex_once(frame, label="First BIND frame (example)")
                    first = False

                ser.write(frame)
                time.sleep(period)

    print("\nFinished scan. If it bound during one of the attempts, power-cycle RX and run normal-mode code.")

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    ser.close()
    print("Serial closed.")


AFHDS2A binding scan… put RX in bind mode, then run this. Ctrl+C to stop.

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=0 (sending 5.0s)
First BIND frame (example) (26 bytes):
55 DC 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=1 (sending 5.0s)

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=2 (sending 5.0s)

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=3 (sending 5.0s)

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=4 (sending 5.0s)

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=5 (sending 5.0s)

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=6 (sending 5.0s)

Trying AFHDS2A bind: sub=PPM_IBUS(1) rx_num=7 (sending 5.0s)

Trying AFHDS2A bind: sub=PWM_IBUS(0) rx_num=0 (sending 5.0s)

Trying AFHDS2A bind: sub=PWM_IBUS(0) rx_num=1 (sending 5.0s)

Trying AFHDS2A bind: sub=PWM_IBUS(0) rx_num=2 (sending 5.0s)

Trying AFHDS2A bind: sub=PWM_IBUS(0) rx_num=3 (sending 5.0s)

Trying AFHDS2A bind: sub=PWM_IBUS(0) rx_num=4 (sending 5.0s)

Tr

In [34]:
import time
import serial

def pack_channels_11bit(ch):
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        if not (0 <= v <= 2047):
            raise ValueError("Channel values must be 0..2047")
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,
    rx_num=0,
    type_value=0,
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF
    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    print(f"{label} ({len(frame)} bytes):")
    print(" ".join(f"{b:02X}" for b in frame))

# ---- Serial: MULTI = 100000 baud, 8E2
PORT = "/dev/ttyUSB0"
BAUD = 100000
ser = serial.Serial(
    PORT, BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

PROTO_AFHDS2A = 28

SUBMODES = [
    ("PPM_IBUS",   1),
    ("PWM_IBUS",   0),
    ("PPM_SBUS",   3),
    ("PWM_SBUS",   2),
    ("PPM_IBUS16", 5),
    ("PWM_IBUS16", 4),
    ("PPM_SBUS16", 7),
    ("PWM_SBUS16", 6),
]

# Try lots of receiver numbers; AFHDS2A is sensitive to rx_num usage. :contentReference[oaicite:6]{index=6}
RX_NUMS = list(range(0, 16))

option_value = 0
frame_hz = 50
period = 1.0 / frame_hz
channels = [1024] * 16

print("AFHDS2A bind scan…")
print("Put the RockSta receiver into bind mode, keep it close to the module, then run this cell.")
print("Ctrl+C to stop.")

try:
    bind_seconds_each = 6.0
    printed_example = False

    for (name, type_value) in SUBMODES:
        for rx_num in RX_NUMS:
            print(f"\nTrying: sub={name} type={type_value} rx_num={rx_num}")

            t0 = time.time()
            while time.time() - t0 < bind_seconds_each:
                frame = make_multi_v1_frame_channels(
                    sub_protocol=PROTO_AFHDS2A,
                    rx_num=rx_num,
                    type_value=type_value,
                    option_value=option_value,
                    bind=True,
                    autobind=True,   # often needed for AFHDS2A workflows :contentReference[oaicite:7]{index=7}
                    channels=channels
                )

                if not printed_example:
                    print_frame_hex_once(frame, "Example BIND frame (first sent)")
                    printed_example = True

                ser.write(frame)
                time.sleep(period)

    print("\nScan finished. If it bound during one attempt, power-cycle the receiver and run your normal-mode streamer.")

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    ser.close()
    print("Serial closed.")


AFHDS2A bind scan…
Put the RockSta receiver into bind mode, keep it close to the module, then run this cell.
Ctrl+C to stop.

Trying: sub=PPM_IBUS type=1 rx_num=0
Example BIND frame (first sent) (26 bytes):
55 DC 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80

Trying: sub=PPM_IBUS type=1 rx_num=1

Trying: sub=PPM_IBUS type=1 rx_num=2

Trying: sub=PPM_IBUS type=1 rx_num=3

Trying: sub=PPM_IBUS type=1 rx_num=4

Trying: sub=PPM_IBUS type=1 rx_num=5

Trying: sub=PPM_IBUS type=1 rx_num=6

Trying: sub=PPM_IBUS type=1 rx_num=7

Trying: sub=PPM_IBUS type=1 rx_num=8

Trying: sub=PPM_IBUS type=1 rx_num=9

Trying: sub=PPM_IBUS type=1 rx_num=10

Trying: sub=PPM_IBUS type=1 rx_num=11

Trying: sub=PPM_IBUS type=1 rx_num=12

Trying: sub=PPM_IBUS type=1 rx_num=13

Trying: sub=PPM_IBUS type=1 rx_num=14

Trying: sub=PPM_IBUS type=1 rx_num=15

Trying: sub=PWM_IBUS type=0 rx_num=0

Trying: sub=PWM_IBUS type=0 rx_num=1

Trying: sub=PWM_IBUS type=0 rx_num=2

Trying: sub=PWM_IBUS typ

In [36]:
import time
import serial

# ----------------------------
# MULTI Serial: channel packing (16ch x 11-bit into 22 bytes)
# ----------------------------
def pack_channels_11bit(ch):
    """
    Pack 16 channels (11-bit each, 0..2047) into 22 bytes, SBUS-style bit packing.
    """
    if len(ch) != 16:
        raise ValueError("Need exactly 16 channels")
    for v in ch:
        if not (0 <= v <= 2047):
            raise ValueError("Channel values must be 0..2047")

    out = bytearray(22)
    bitpos = 0
    for v in ch:
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def print_frame_hex_once(frame, label="TX frame"):
    print(f"{label} ({len(frame)} bytes):")
    print(" ".join(f"{b:02X}" for b in frame))

# ----------------------------
# MULTI Serial V2 frame builder (per Multiprotocol.h)
# ----------------------------
def make_multi_serial_v2_channel_frame(
    protocol,            # 0..255 (this is what Multiprotocol.h calls "sub_protocol" in Stream[1])
    rx_num=0,             # 0..63 (V2 extends RX number using Stream[26] bits 4..5)
    type_value=0,         # 0..7 (Stream[2] bits 4..6) = protocol-specific subtype
    option_value=0,       # Stream[3] signed int8 on wire (we send as byte)
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    telemetry_invert=False,
    disable_telemetry=False,
    disable_ch_mapping=False,
    extra=b"",            # Stream[27..35] optional protocol data (0..9 bytes). AFHDS2A typically uses none.
    channels=None
):
    """
    Serial V2: length = 27..36 bytes
      Stream[0] header: 0x55/0x54 (channels) depending on protocol bit5
      Stream[1] bits0..4 = protocol bits0..4 + flags (bind/range/autobind)
      Stream[2] rx_num bits0..3 + type_value<<4 + low_power bit7
      Stream[3] option_protocol (int8)
      Stream[4..25] packed channels (22 bytes)
      Stream[26] protocol bits6..7 | rx_num bits4..5 | telemetry flags
      Stream[27..] extra protocol data (0..9 bytes)
    """
    if channels is None:
        channels = [1024] * 16
    if not (0 <= protocol <= 255):
        raise ValueError("protocol must be 0..255")
    if not (0 <= rx_num <= 63):
        raise ValueError("rx_num must be 0..63")
    if not (0 <= type_value <= 7):
        raise ValueError("type_value must be 0..7")
    if len(extra) > 9:
        raise ValueError("extra must be 0..9 bytes (Stream[27..35])")

    # Stream[0] header depends on protocol bit5 (values 0..31 => 0x55, 32..63 => 0x54, etc.)
    # In V2, higher protocol bits 6..7 are carried in Stream[26].
    header = 0x54 if (protocol & 0x20) else 0x55  # channels frame

    # Stream[1] = protocol bits0..4 + flags
    b1 = protocol & 0x1F
    if bind:       b1 |= 0x80
    if rangecheck: b1 |= 0x20
    if autobind:   b1 |= 0x40

    # Stream[2] = rx_num bits0..3 + type<<4 + low power bit7
    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    # Stream[3] = option byte
    b3 = option_value & 0xFF

    # Stream[4..25]
    ch_payload = pack_channels_11bit(channels)

    # Stream[26]:
    #  - protocol bits6..7 go into bits6..7
    #  - rx_num bits4..5 go into bits4..5
    #  - bit3 telemetry invert
    #  - bit1 disable telemetry
    #  - bit0 disable CH mapping
    b26 = 0
    b26 |= (protocol & 0xC0)               # bits6..7 already aligned
    b26 |= ((rx_num >> 4) & 0x03) << 4     # bits4..5
    if telemetry_invert:
        b26 |= 0x08
    if disable_telemetry:
        b26 |= 0x02
    if disable_ch_mapping:
        b26 |= 0x01

    frame = bytes([header, b1, b2, b3]) + ch_payload + bytes([b26]) + extra
    return frame

# ------------------------
# Serial setup (MULTI serial: 100000 baud, 8E2)
# ------------------------
PORT = "/dev/ttyUSB0"   # change if needed
BAUD = 100000

PROTO_AFHDS2A = 28

# AFHDS2A sub-modes are carried in type_value (Stream[2] bits 4..6)
# Common mapping used by many MULTI builds:
PWM_IBUS     = 0
PPM_IBUS     = 1
PWM_SBUS     = 2
PPM_SBUS     = 3
PWM_IBUS16   = 4
PPM_IBUS16   = 5
PWM_SBUS16   = 6
PPM_SBUS16   = 7

type_value = PPM_IBUS      # "PPM mode"
rx_num = 0                 # try 0..63 in V2 if needed
option_value = 0           # safe default
frame_hz = 50
period = 1.0 / frame_hz

channels = [1024] * 16
t0 = time.time()

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

print("AFHDS2A Serial V2 normal streaming… Ctrl+C to stop")

try:
    # Print one frame in hex (once)
    sample = make_multi_serial_v2_channel_frame(
        protocol=PROTO_AFHDS2A,
        rx_num=rx_num,
        type_value=type_value,
        option_value=option_value,
        bind=False,
        autobind=False,
        disable_telemetry=False,
        disable_ch_mapping=False,
        channels=channels
    )
    print_frame_hex_once(sample, label="Sample V2 channel frame")

    while True:
        elapsed = time.time() - t0
        phase = (elapsed % 2.0) / 2.0
        channels[0] = int(204 + phase * (1843 - 204))  # CH1 sweep

        frame = make_multi_serial_v2_channel_frame(
            protocol=PROTO_AFHDS2A,
            rx_num=rx_num,
            type_value=type_value,
            option_value=option_value,
            bind=False,
            autobind=False,
            channels=channels
        )
        ser.write(frame)
        time.sleep(period)

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    ser.close()
    print("Serial closed.")


AFHDS2A Serial V2 normal streaming… Ctrl+C to stop
Sample V2 channel frame (27 bytes):
55 1C 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 00

Stopped (Ctrl+C).
Serial closed.


## Try to bind to AFHDS2A with serial v2 protocol

In [43]:
import time
import serial

def pack_channels_11bit(ch):
    if len(ch) != 16:
        raise ValueError("Need exactly 16 channels")
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        if not (0 <= v <= 2047):
            raise ValueError("Channel values must be 0..2047")
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_serial_v2_channel_frame(
    protocol,
    rx_num=0,                 # 0..63
    type_value=0,             # 0..7  (this is the AFHDS2A "sub-mode")
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    telemetry_invert=False,
    disable_telemetry=False,
    disable_ch_mapping=False,
    extra=b"",                # 0..9 bytes
    channels=None
):
    if channels is None:
        channels = [1024] * 16
    if not (0 <= protocol <= 255):
        raise ValueError("protocol must be 0..255")
    if not (0 <= rx_num <= 63):
        raise ValueError("rx_num must be 0..63")
    if not (0 <= type_value <= 7):
        raise ValueError("type_value must be 0..7")
    if len(extra) > 9:
        raise ValueError("extra must be 0..9 bytes")

    header = 0x54 if (protocol & 0x20) else 0x55

    b1 = protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF
    ch_payload = pack_channels_11bit(channels)

    b26 = 0
    b26 |= (protocol & 0xC0)               # protocol bits 6..7
    b26 |= ((rx_num >> 4) & 0x03) << 4     # rx_num bits 4..5
    if telemetry_invert:
        b26 |= 0x08
    if disable_telemetry:
        b26 |= 0x02
    if disable_ch_mapping:
        b26 |= 0x01

    return bytes([header, b1, b2, b3]) + ch_payload + bytes([b26]) + extra

def hexdump(b: bytes) -> str:
    return " ".join(f"{x:02X}" for x in b)

# ------------------------
# CONFIG
# ------------------------
PORT = "/dev/ttyUSB0"
BAUD = 100000
PROTO_AFHDS2A = 28

SUBMODES = [
    ("PWM_IBUS",   0),
    ("PPM_IBUS",   1),
    ("PWM_SBUS",   2),
    ("PPM_SBUS",   3),
    ("PWM_IBUS16", 4),
    ("PPM_IBUS16", 5),
    ("PWM_SBUS16", 6),
    ("PPM_SBUS16", 7),
]

# Try many receiver numbers; V2 supports 0..63
RX_NUMS = list(range(0, 3))

frame_hz = 50
period = 1.0 / frame_hz
bind_seconds_each = 3.0

channels = [1024] * 16
option_value = 0

print("AFHDS2A Serial V2 bind scan.")
print("1) Put receiver into bind mode.")
print("2) Keep it close to the module.")
print("3) Run this cell. Ctrl+C to stop.\n")

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

try:
    printed = False

    for autobind in (True, False):
        print(f"\n=== autobind = {autobind} ===")
        for (name, type_value) in SUBMODES:
            for rx_num in RX_NUMS:
                print(f"Trying: {name} (type={type_value})  rx_num={rx_num}")

                t0 = time.time()
                while time.time() - t0 < bind_seconds_each:
                    frame = make_multi_serial_v2_channel_frame(
                        protocol=PROTO_AFHDS2A,
                        rx_num=rx_num,
                        type_value=type_value,
                        option_value=option_value,
                        bind=True,
                        autobind=autobind,
                        channels=channels
                    )
                    if not printed:
                        print("\nExample bind frame:")
                        print(hexdump(frame))
                        printed = True

                    ser.write(frame)
                    time.sleep(period)

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    ser.close()
    print("Serial closed.")


AFHDS2A Serial V2 bind scan.
1) Put receiver into bind mode.
2) Keep it close to the module.
3) Run this cell. Ctrl+C to stop.


=== autobind = True ===
Trying: PWM_IBUS (type=0)  rx_num=0

Example bind frame:
55 DC 00 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 00
Trying: PWM_IBUS (type=0)  rx_num=1
Trying: PWM_IBUS (type=0)  rx_num=2
Trying: PPM_IBUS (type=1)  rx_num=0
Trying: PPM_IBUS (type=1)  rx_num=1
Trying: PPM_IBUS (type=1)  rx_num=2
Trying: PWM_SBUS (type=2)  rx_num=0
Trying: PWM_SBUS (type=2)  rx_num=1
Trying: PWM_SBUS (type=2)  rx_num=2
Trying: PPM_SBUS (type=3)  rx_num=0
Trying: PPM_SBUS (type=3)  rx_num=1
Trying: PPM_SBUS (type=3)  rx_num=2
Trying: PWM_IBUS16 (type=4)  rx_num=0
Trying: PWM_IBUS16 (type=4)  rx_num=1
Trying: PWM_IBUS16 (type=4)  rx_num=2
Trying: PPM_IBUS16 (type=5)  rx_num=0
Trying: PPM_IBUS16 (type=5)  rx_num=1
Trying: PPM_IBUS16 (type=5)  rx_num=2
Trying: PWM_SBUS16 (type=6)  rx_num=0
Trying: PWM_SBUS16 (type=6)  rx_num=1
Trying: P

AFHDS2A V2 bind scan (likely-first). Ctrl+C to stop.

Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=0

Example bind frame:
 55 DC 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 00 

Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=1
Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=2
Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=3

Stopped.
Serial closed.


# Flysky Protocol with Serial V2

## FlySky bind + CH1 simulation (Serial V2)

In [42]:
import time
import serial

# ----------------------------
# MULTI Serial: pack 16ch x 11-bit into 22 bytes
# ----------------------------
def pack_channels_11bit(ch):
    if len(ch) != 16:
        raise ValueError("Need exactly 16 channels")
    for v in ch:
        if not (0 <= v <= 2047):
            raise ValueError("Channel values must be 0..2047")

    out = bytearray(22)
    bitpos = 0
    for v in ch:
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def hexdump(b: bytes) -> str:
    return " ".join(f"{x:02X}" for x in b)

# ----------------------------
# MULTI Serial V2 frame builder (per your Multiprotocol.h)
# ----------------------------
def make_multi_serial_v2_channel_frame(
    protocol,               # 0..255 (protocol number, e.g., 1 for FlySky)
    rx_num=0,                # 0..63
    type_value=0,            # 0..7  (FlySky subtype: 0=Flysky, 1=V9x9, 2=V6x6, 3=V912, 4=CX20)
    option_value=0,          # Stream[3]
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    telemetry_invert=False,
    disable_telemetry=False,
    disable_ch_mapping=False,
    extra=b"",               # Stream[27..35] optional (0..9 bytes) - unused for FlySky
    channels=None
):
    if channels is None:
        channels = [1024] * 16
    if not (0 <= protocol <= 255):
        raise ValueError("protocol must be 0..255")
    if not (0 <= rx_num <= 63):
        raise ValueError("rx_num must be 0..63")
    if not (0 <= type_value <= 7):
        raise ValueError("type_value must be 0..7")
    if len(extra) > 9:
        raise ValueError("extra must be 0..9 bytes")

    # Stream[0]: header 0x55 for protocols with bit5=0; 0x54 if bit5=1
    header = 0x54 if (protocol & 0x20) else 0x55

    # Stream[1]: protocol bits0..4 + bind/autobind/range flags
    b1 = protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    # Stream[2]: rx_num bits0..3 + type<<4 + low_power bit7
    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    # Stream[3]: option byte
    b3 = option_value & 0xFF

    # Stream[4..25]: channels packed
    ch_payload = pack_channels_11bit(channels)

    # Stream[26]:
    #  - protocol bits6..7 in bits6..7
    #  - rx_num bits4..5 in bits4..5
    #  - bit3 telemetry invert
    #  - bit1 disable telemetry
    #  - bit0 disable ch mapping
    b26 = 0
    b26 |= (protocol & 0xC0)
    b26 |= ((rx_num >> 4) & 0x03) << 4
    if telemetry_invert:
        b26 |= 0x08
    if disable_telemetry:
        b26 |= 0x02
    if disable_ch_mapping:
        b26 |= 0x01

    return bytes([header, b1, b2, b3]) + ch_payload + bytes([b26]) + extra

# ------------------------
# Serial setup: MULTI uses 100000 8E2
# ------------------------
PORT = "/dev/ttyUSB0"   # change if needed
BAUD = 100000

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

# ------------------------
# FlySky (AFHDS v1) settings
# ------------------------
PROTO_FLYSKY = 1
FLYSKY_SUBTYPE = 0   # 0 = Flysky AFHDS (classic). Others: 1 V9x9, 2 V6x6, 3 V912, 4 CX20

rx_num = 0           # usually 0..15 is enough, but V2 allows 0..63
option_value = 0

frame_hz = 50
period = 1.0 / frame_hz

channels = [1024] * 16

def bind_then_run(bind_seconds=6.0):
    print("FlySky Serial V2: binding… (Ctrl+C to stop)")
    t0 = time.time()

    # Print one bind frame
    bind_frame = make_multi_serial_v2_channel_frame(
        protocol=PROTO_FLYSKY,
        rx_num=rx_num,
        type_value=FLYSKY_SUBTYPE,
        option_value=option_value,
        bind=True,
        autobind=False,
        channels=channels
    )
    print("BIND frame:", hexdump(bind_frame))

    # Bind phase
    while time.time() - t0 < bind_seconds:
        ser.write(bind_frame)
        time.sleep(period)

    print("Bind phase done. Switching to normal mode (CH1 sweep). Ctrl+C to stop.")
    t1 = time.time()

    # Normal phase
    while True:
        elapsed = time.time() - t1
        phase = (elapsed % 2.0) / 2.0
        channels[0] = int(204 + phase * (1843 - 204))  # CH1 sweep

        frame = make_multi_serial_v2_channel_frame(
            protocol=PROTO_FLYSKY,
            rx_num=rx_num,
            type_value=FLYSKY_SUBTYPE,
            option_value=option_value,
            bind=False,
            autobind=False,
            channels=channels
        )
        ser.write(frame)
        time.sleep(period)

try:
    bind_then_run(bind_seconds=6.0)
except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")
finally:
    # send a couple neutral frames, then close
    neutral = [1024] * 16
    for _ in range(3):
        ser.write(make_multi_serial_v2_channel_frame(
            protocol=PROTO_FLYSKY,
            rx_num=rx_num,
            type_value=FLYSKY_SUBTYPE,
            option_value=option_value,
            bind=False,
            channels=neutral
        ))
        time.sleep(0.02)
    ser.close()
    print("Serial closed.")


FlySky Serial V2: binding… (Ctrl+C to stop)
BIND frame: 55 81 00 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 00
Bind phase done. Switching to normal mode (CH1 sweep). Ctrl+C to stop.

Stopped (Ctrl+C).
Serial closed.


In [48]:
import time
import serial

def pack_channels_11bit(ch):
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_serial_v2_channel_frame(
    protocol,
    rx_num=0,
    type_value=0,
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    telemetry_invert=False,
    disable_telemetry=False,
    disable_ch_mapping=False,
    extra=b"",
    channels=None
):
    if channels is None:
        channels = [1024]*16

    header = 0x54 if (protocol & 0x20) else 0x55

    b1 = protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF
    ch_payload = pack_channels_11bit(channels)

    b26 = 0
    b26 |= (protocol & 0xC0)
    b26 |= ((rx_num >> 4) & 0x03) << 4
    if telemetry_invert:
        b26 |= 0x08
    if disable_telemetry:
        b26 |= 0x02
    if disable_ch_mapping:
        b26 |= 0x01

    return bytes([header, b1, b2, b3]) + ch_payload + bytes([b26]) + extra

def hexdump(b: bytes) -> str:
    return " ".join(f"{x:02X}" for x in b)

PORT = "/dev/ttyUSB0"
BAUD = 100000

PROTO_AFHDS2A = 28

# Likely-first ordering (most common are IBUS variants)
SUBMODES = [
    ("PPM_IBUS", 1),
    ("PWM_IBUS", 0),
    ("PPM_IBUS16", 5),
    ("PWM_IBUS16", 4),
    ("PPM_SBUS", 3),
    ("PWM_SBUS", 2),
    ("PPM_SBUS16", 7),
    ("PWM_SBUS16", 6),
]

# Try the most likely rx_num values first, then expand
RX_NUMS = list(range(0, 16, 4)) + list(range(16, 64, 16))  # quick coverage

OPTION_VALUES = [0, 10, 20]   # conservative + common alternates
AUTOFLAGS = [True, False]

frame_hz = 50
period = 1.0 / frame_hz
bind_seconds_each = 2.0

channels = [1024]*16

ser = serial.Serial(
    PORT, BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

print("AFHDS2A V2 bind scan (likely-first). Ctrl+C to stop.\n")

try:
    printed = False
    for autobind in AUTOFLAGS:
        for opt in OPTION_VALUES:
            for (name, type_value) in SUBMODES:
                for rx_num in RX_NUMS:
                    print(f"Trying: autobind={autobind} opt={opt} sub={name} type={type_value} rx_num={rx_num}")

                    t0 = time.time()
                    while time.time() - t0 < bind_seconds_each:
                        frame = make_multi_serial_v2_channel_frame(
                            protocol=PROTO_AFHDS2A,
                            rx_num=rx_num,
                            type_value=type_value,
                            option_value=opt,
                            bind=True,
                            autobind=autobind,
                            channels=channels
                        )
                        if not printed:
                            print("\nExample bind frame:\n", hexdump(frame), "\n")
                            printed = True

                        ser.write(frame)
                        time.sleep(period)

except KeyboardInterrupt:
    print("\nStopped.")
finally:
    ser.close()
    print("Serial closed.")


AFHDS2A V2 bind scan (likely-first). Ctrl+C to stop.

Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=0

Example bind frame:
 55 DC 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 00 

Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=4
Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=8
Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=12
Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=16
Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=32
Trying: autobind=True opt=0 sub=PPM_IBUS type=1 rx_num=48
Trying: autobind=True opt=0 sub=PWM_IBUS type=0 rx_num=0
Trying: autobind=True opt=0 sub=PWM_IBUS type=0 rx_num=4
Trying: autobind=True opt=0 sub=PWM_IBUS type=0 rx_num=8
Trying: autobind=True opt=0 sub=PWM_IBUS type=0 rx_num=12
Trying: autobind=True opt=0 sub=PWM_IBUS type=0 rx_num=16
Trying: autobind=True opt=0 sub=PWM_IBUS type=0 rx_num=32
Trying: autobind=True opt=0 sub=PWM_IBUS type=0 rx_num=48
Trying: autobind=True opt=0 sub=PPM

# Try Protocols

In [4]:
import time
import re
from pathlib import Path
from dataclasses import dataclass
import serial

import pprint

In [2]:
# ===== USER CONFIG =====
HEADER_PATH = "Multiprotocol.h"   # re-upload this file first
SERIAL_PORT = "/dev/ttyUSB0"          # FTDI device
BAUDRATE = 100000                     # MULTI protocol
FRAME_RATE_HZ = 50                    # send rate
DWELL_SECONDS = 2.0                   # per protocol attempt
RXNUM = 0                             # receiver number
LOW_POWER = False
OPTION_PROTOCOL = 0                  # signed -128..127


In [12]:
ENUM_RE = re.compile(r"enum\s+([A-Za-z0-9_]+)\s*\{(.*?)\};", re.S)
ENUM_ENTRY_RE = re.compile(r"^\s*([A-Za-z0-9_]+)\s*=\s*([0-9]+)\s*,?\s*$")
PROTO_RE = re.compile(r"^\s*PROTO_([A-Za-z0-9_]+)\s*=\s*([0-9]+)\s*,", re.M)

def parse_header(path):
    text = Path(path).read_text(errors="ignore")

    enums = {}
    for m in ENUM_RE.finditer(text):
        name = m.group(1)
        body = m.group(2)
        values = {}
        for line in body.splitlines():
            m2 = ENUM_ENTRY_RE.match(line.strip())
            if m2:
                values[m2.group(1)] = int(m2.group(2))
        if values:
            enums[name] = values

    protos = {m.group(1): int(m.group(2)) for m in PROTO_RE.finditer(text)}
    return enums, protos

enums, protos = parse_header(HEADER_PATH)

print("Protocols:", protos)
#pprint.pp(protos)
print("Enums:", enums.keys())


Protocols: {'PROTOLIST': 0, 'FLYSKY': 1, 'HUBSAN': 2, 'FRSKYD': 3, 'HISKY': 4, 'V2X2': 5, 'DSM': 6, 'DEVO': 7, 'YD717': 8, 'KN': 9, 'SYMAX': 10, 'SLT': 11, 'CX10': 12, 'CG023': 13, 'BAYANG': 14, 'FRSKYX': 15, 'ESKY': 16, 'MT99XX': 17, 'MJXQ': 18, 'SHENQI': 19, 'FY326': 20, 'FUTABA': 21, 'J6PRO': 22, 'FQ777': 23, 'ASSAN': 24, 'FRSKYV': 25, 'HONTAI': 26, 'OPENLRS': 27, 'AFHDS2A': 28, 'Q2X2': 29, 'WK2x01': 30, 'Q303': 31, 'GW008': 32, 'DM002': 33, 'CABELL': 34, 'ESKY150': 35, 'H8_3D': 36, 'CORONA': 37, 'CFLIE': 38, 'HITEC': 39, 'WFLY': 40, 'BUGS': 41, 'BUGSMINI': 42, 'TRAXXAS': 43, 'NCC1701': 44, 'E01X': 45, 'V911S': 46, 'GD00X': 47, 'V761': 48, 'KF606': 49, 'REDPINE': 50, 'POTENSIC': 51, 'ZSX': 52, 'HEIGHT': 53, 'SCANNER': 54, 'FRSKY_RX': 55, 'AFHDS2A_RX': 56, 'HOTT': 57, 'FX': 58, 'BAYANG_RX': 59, 'PELIKAN': 60, 'EAZYRC': 61, 'XK': 62, 'XN297DUMP': 63, 'FRSKYX2': 64, 'FRSKY_R9': 65, 'PROPEL': 66, 'FRSKYL': 67, 'SKYARTEC': 68, 'ESKY150V2': 69, 'DSM_RX': 70, 'JJRC345': 71, 'Q90C': 72, 'KY

In [13]:
def pack_16ch_11bit(ch):
    bitbuf = 0
    bits = 0
    out = bytearray()
    for v in ch:
        bitbuf |= (v & 0x7FF) << bits
        bits += 11
        while bits >= 8:
            out.append(bitbuf & 0xFF)
            bitbuf >>= 8
            bits -= 8
    return bytes(out[:22])


def build_multi_frame(protocol, subtype, bind=True):
    channels = [1024] * 16

    proto_bit5 = (protocol >> 5) & 1
    stream0 = 0x54 if proto_bit5 else 0x55

    stream1 = (protocol & 0x1F) | (0x80 if bind else 0)
    stream2 = (RXNUM & 0x0F) | ((subtype & 0x07) << 4)
    stream3 = OPTION_PROTOCOL & 0xFF

    payload = pack_16ch_11bit(channels)

    stream26 = (protocol & 0xC0) | ((RXNUM >> 4) << 4) | 0x02  # disable telemetry

    return bytes([stream0, stream1, stream2, stream3]) + payload + bytes([stream26])


In [14]:
@dataclass
class Attempt:
    proto_name: str
    proto_id: int
    subtype_name: str
    subtype_id: int


attempts = []

# Flysky AFHDS
if "FLYSKY" in protos and "Flysky" in enums:
    for name, sid in enums["Flysky"].items():
        attempts.append(Attempt("FLYSKY", protos["FLYSKY"], name, sid))

# AFHDS2A
if "AFHDS2A" in protos and "AFHDS2A" in enums:
    for name, sid in enums["AFHDS2A"].items():
        attempts.append(Attempt("AFHDS2A", protos["AFHDS2A"], name, sid))

attempts.sort(key=lambda a: (a.proto_id, a.subtype_id))

print(f"Total attempts: {len(attempts)}")
for a in attempts:
    print(a)


Total attempts: 13
Attempt(proto_name='FLYSKY', proto_id=1, subtype_name='Flysky', subtype_id=0)
Attempt(proto_name='FLYSKY', proto_id=1, subtype_name='V9X9', subtype_id=1)
Attempt(proto_name='FLYSKY', proto_id=1, subtype_name='V6X6', subtype_id=2)
Attempt(proto_name='FLYSKY', proto_id=1, subtype_name='V912', subtype_id=3)
Attempt(proto_name='FLYSKY', proto_id=1, subtype_name='CX20', subtype_id=4)
Attempt(proto_name='AFHDS2A', proto_id=28, subtype_name='PWM_IBUS', subtype_id=0)
Attempt(proto_name='AFHDS2A', proto_id=28, subtype_name='PPM_IBUS', subtype_id=1)
Attempt(proto_name='AFHDS2A', proto_id=28, subtype_name='PWM_SBUS', subtype_id=2)
Attempt(proto_name='AFHDS2A', proto_id=28, subtype_name='PPM_SBUS', subtype_id=3)
Attempt(proto_name='AFHDS2A', proto_id=28, subtype_name='PWM_IB16', subtype_id=4)
Attempt(proto_name='AFHDS2A', proto_id=28, subtype_name='PPM_IB16', subtype_id=5)
Attempt(proto_name='AFHDS2A', proto_id=28, subtype_name='PWM_SB16', subtype_id=6)
Attempt(proto_name='AFHDS

In [18]:
ser = serial.Serial(
    port=SERIAL_PORT,
    baudrate=BAUDRATE,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0.1,
    write_timeout=0.1,
)

print("Serial opened:", ser.name)


Serial opened: /dev/ttyUSB0


In [19]:
interval = 1.0 / FRAME_RATE_HZ

try:
    for idx, a in enumerate(attempts, 1):
        print(f"\n[{idx}/{len(attempts)}] "
              f"{a.proto_name}:{a.subtype_name} "
              f"(proto={a.proto_id}, subtype={a.subtype_id})")

        frame = build_multi_frame(a.proto_id, a.subtype_id, bind=True)
        print("Frame:", frame.hex(" "))

        end_time = time.time() + DWELL_SECONDS
        while time.time() < end_time:
            ser.write(frame)
            time.sleep(interval)

except KeyboardInterrupt:
    print("\nCtrl+C detected — stopping gracefully.")

finally:
    ser.close()
    print("Serial closed.")



[1/13] FLYSKY:Flysky (proto=1, subtype=0)
Frame: 55 81 00 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 02

[2/13] FLYSKY:V9X9 (proto=1, subtype=1)
Frame: 55 81 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 02

[3/13] FLYSKY:V6X6 (proto=1, subtype=2)
Frame: 55 81 20 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 02

[4/13] FLYSKY:V912 (proto=1, subtype=3)
Frame: 55 81 30 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 02

[5/13] FLYSKY:CX20 (proto=1, subtype=4)
Frame: 55 81 40 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 02

[6/13] AFHDS2A:PWM_IBUS (proto=28, subtype=0)
Frame: 55 9c 00 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 02

[7/13] AFHDS2A:PPM_IBUS (proto=28, subtype=1)
Frame: 55 9c 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80 02

[8/13] AFHDS2A:PWM_SBUS (proto=28, subtype=2)
Frame: 55 9c 20 00 00 04 20 00 01 08